# 02 — Terraform Basics Demo

Companion notebook to `03-infrastructure-as-code-with-terraform.md`. This notebook is markdown-heavy on purpose: Terraform itself isn't executed here (that would require the `terraform` CLI and a real Azure subscription, which this offline course deliberately avoids). Instead, the `.tf` configuration is shown as **text** for reading and discussion, and the one part that *is* real, runnable Python is a small parser for simplified Terraform **plan output** — the artifact you'd actually review in a pull request before approving an `apply`.

## 1. Terraform workflow, in one picture

```
  write/edit .tf files (providers, resources, variables, modules)
         |
         v
  terraform init      # downloads providers, configures the remote state backend
         |
         v
  terraform plan      # computes the diff between desired config and current state
         |            # (no changes applied yet -- this is what a reviewer reads in a PR)
         v
  [ review / approval gate ]
         |
         v
  terraform apply     # executes the plan: calls the provider API, updates state
```

In a CI/CD pipeline (chapter 02), `plan` typically runs on every pull request so reviewers see the infrastructure diff alongside the code diff, and `apply` runs after merge and after an approval gate on the target Environment — mirroring the build-then-gated-deploy pattern used for application code.

## 2. Example `.tf` configuration (shown as text, not executed)

This is the same Azure App Service + Key Vault example from chapter 03 — the infrastructure shape underneath both the chatbot service (course 01) and the document-uploader service (course 03). It is included here purely for reference; running it for real would require the `terraform` CLI, the `azurerm` provider, and credentials for a real Azure subscription, none of which this notebook has or needs.

```hcl
resource "azurerm_resource_group" "rg" {
  name     = "rg-${var.app_name}-${var.environment}"
  location = var.location
}

resource "azurerm_service_plan" "plan" {
  name                = "asp-${var.app_name}-${var.environment}"
  resource_group_name = azurerm_resource_group.rg.name
  location            = azurerm_resource_group.rg.location
  os_type             = "Linux"
  sku_name            = var.sku_name
}

resource "azurerm_linux_web_app" "app" {
  name                = "${var.app_name}-${var.environment}"
  resource_group_name = azurerm_resource_group.rg.name
  location            = azurerm_resource_group.rg.location
  service_plan_id     = azurerm_service_plan.plan.id

  identity {
    type = "SystemAssigned"
  }

  app_settings = {
    KEY_VAULT_URI = azurerm_key_vault.kv.vault_uri
  }
}

resource "azurerm_key_vault" "kv" {
  name                = "kv-${var.app_name}-${var.environment}"
  resource_group_name = azurerm_resource_group.rg.name
  location            = azurerm_resource_group.rg.location
  tenant_id           = var.tenant_id
  sku_name            = "standard"
}
```

Applying this for the first time in a brand-new environment would create 4 explicit resources (resource group, service plan, web app, key vault) plus the access policy shown in the chapter — which is exactly the kind of count the plan-summary parser below is meant to report.

## 3. Parsing a simplified Terraform plan summary

Real `terraform show -json` output is verbose. The simplified structure below keeps just the part that matters for a plan review: a list of `resource_changes`, each with a `change.actions` list (`["create"]`, `["update"]`, `["delete"]`, `["no-op"]`, or `["create", "delete"]` for a replace). This is plain `dict`/`list`/`json` handling — no Terraform installation required.

In [1]:
import json

sample_plan_output = {
    "resource_changes": [
        {"address": "azurerm_resource_group.rg", "change": {"actions": ["create"]}},
        {"address": "azurerm_service_plan.plan", "change": {"actions": ["create"]}},
        {"address": "azurerm_linux_web_app.app", "change": {"actions": ["create"]}},
        {"address": "azurerm_key_vault.kv", "change": {"actions": ["create"]}},
        {"address": "azurerm_key_vault_access_policy.app_access", "change": {"actions": ["create"]}},
        {"address": "azurerm_linux_web_app.app.app_settings", "change": {"actions": ["update"]}},
        {"address": "azurerm_service_plan.old_plan", "change": {"actions": ["delete"]}},
    ]
}

print(json.dumps(sample_plan_output, indent=2))

{
  "resource_changes": [
    {
      "address": "azurerm_resource_group.rg",
      "change": {
        "actions": [
          "create"
        ]
      }
    },
    {
      "address": "azurerm_service_plan.plan",
      "change": {
        "actions": [
          "create"
        ]
      }
    },
    {
      "address": "azurerm_linux_web_app.app",
      "change": {
        "actions": [
          "create"
        ]
      }
    },
    {
      "address": "azurerm_key_vault.kv",
      "change": {
        "actions": [
          "create"
        ]
      }
    },
    {
      "address": "azurerm_key_vault_access_policy.app_access",
      "change": {
        "actions": [
          "create"
        ]
      }
    },
    {
      "address": "azurerm_linux_web_app.app.app_settings",
      "change": {
        "actions": [
          "update"
        ]
      }
    },
    {
      "address": "azurerm_service_plan.old_plan",
      "change": {
        "actions": [
          "delete"
        ]
      }
    }
 

In [2]:
def summarize_plan(plan):
    """Reduce a simplified Terraform plan JSON structure to add/change/destroy counts.

    Mirrors what a human reviewer scans for at the bottom of a real
    `terraform plan` run: "Plan: N to add, M to change, K to destroy."
    """
    to_add = to_change = to_destroy = 0
    for rc in plan.get("resource_changes", []):
        actions = rc.get("change", {}).get("actions", [])
        if actions == ["no-op"]:
            continue
        elif "create" in actions and "delete" in actions:
            to_change += 1  # replace: counted as a change (destroy+recreate in one step)
        elif "create" in actions:
            to_add += 1
        elif "update" in actions:
            to_change += 1
        elif "delete" in actions:
            to_destroy += 1
    return {"to_add": to_add, "to_change": to_change, "to_destroy": to_destroy}


summary = summarize_plan(sample_plan_output)
print(summary)
print(
    f"Plan: {summary['to_add']} to add, {summary['to_change']} to change, "
    f"{summary['to_destroy']} to destroy."
)

{'to_add': 5, 'to_change': 1, 'to_destroy': 1}
Plan: 5 to add, 1 to change, 1 to destroy.


## 4. Why this matters for the pipeline

The three numbers this function reports — add / change / destroy — are exactly what an approval gate (chapter 04) should surface to a reviewer before a `terraform apply` runs against a shared environment. A plan showing an unexpected `destroy` on a stateful resource (a Key Vault, a database) is the single most important thing to catch here — see `99-Interview-QA.md` Q11 for how this fits into a pull-request review, and Q6 for how remote state locking keeps two concurrent `apply` runs from corrupting each other's view of what already exists.